# TS5 — Estimación de la Densidad Espectral de Potencia (PSD)

**Asignatura:** Análisis y Procesamiento de Señales  
**Autor:** Pedro Joaquín Cannavo  
**Fecha:** Septiembre 2026  

---

En este trabajo se analiza la densidad espectral de potencia de tres señales reales: un electrocardiograma (ECG) durante una prueba de esfuerzo, una señal de pletismografía (PPG) tomada en reposo y un archivo de audio ("La Cucaracha"). 

Para cada señal se implementa el método de estimación espectral de Welch, se exploran distintas combinaciones de parámetros para justificar la elección de la resolución y el suavizado, y finalmente se calcula el ancho de banda efectivo que concentra la mayor parte de la energía.

## Marco Teórico

La Densidad Espectral de Potencia (PSD) nos indica cómo se distribuye la energía de una señal a lo largo de las distintas frecuencias. 

El estimador más directo es el **Periodograma**, que consiste en calcular la FFT de la señal completa y elevar al cuadrado su módulo:

$$\hat{P}_{\text{per}}(f) = \frac{1}{N} \left| \text{FFT}\{x(n)\} \right|^2$$

El principal problema del periodograma es que es un estimador inconsistente: su varianza no disminuye aunque tengamos registros más largos, por lo que el espectro siempre queda con un aspecto de serrucho ruidoso. Además, como aplica implícitamente una ventana rectangular sobre todas las muestras, presenta un desparramo espectral (*spectral leakage*) considerable que puede enmascarar componentes débiles.

El **Método de Welch** soluciona estos problemas aplicando tres pasos:
1. **Segmentación con solapamiento:** Divide la señal de $N$ muestras en $K$ bloques más chicos de longitud $L$ con un solapamiento (*overlap*, típicamente del 50%).
2. **Enventanado suave:** Multiplica cada bloque por una ventana no rectangular $w(n)$ (en este trabajo usamos la ventana Hann), lo que atenúa los bordes y baja fuertemente los lóbulos secundarios del leakage.
3. **Promediado:** Calcula el periodograma modificado de cada segmento y los promedia entre sí para reducir la varianza en un factor proporcional a la cantidad de segmentos $K$:

$$\hat{P}_{xx}^{(\text{Welch})}(f) = \frac{1}{K} \sum_{k=1}^{K} \frac{1}{L \cdot U} \left| \text{FFT}\{ w(n) \cdot x_k(n) \} \right|^2$$

donde $U = \frac{1}{L} \sum_{n=0}^{L-1} w^2(n)$ es el factor de normalización de la potencia de la ventana.

Los parámetros principales a elegir son:
- `nperseg` ($L$): tamaño de cada segmento en muestras. Define la resolución espectral según $\Delta f = f_s / L$. A mayor $L$, mejor resolución (picos más finos), pero menos promedios disponibles y mayor varianza.
- `noverlap`: cantidad de muestras solapadas entre bloques consecutivos. El 50% permite recuperar la información atenuada en los extremos de la ventana y aumentar el número efectivo de promedios $K \approx 1 + \frac{N - L}{L/2}$.
- `scaling='density'`: normaliza el resultado para que la PSD quede expresada en unidades de potencia por Hz ($V^2/\text{Hz}$ o $\text{dB/Hz}$).

### Estimación del Ancho de Banda

Para estimar el ancho de banda efectivo de cada señal se utiliza un criterio acumulativo de energía: se busca la frecuencia de corte $f_{BW}$ hasta la cual se concentra el $99.5\%$ (o $99\%$) de la potencia total contenida en el espectro:

$$\frac{\sum_{f=0}^{f_{BW}} \hat{P}_{xx}(f)}{\sum_{f=0}^{f_s/2} \hat{P}_{xx}(f)} \geq 0.995$$

Este criterio resulta mucho más representativo que el corte a $-3\,\text{dB}$ para señales biomédicas y de audio, donde la energía se reparte en múltiples armónicos y componentes complejas.

## 1. Lectura y visualización temporal de las señales

In [ ]:
import numpy as np
from scipy import signal as sig
import matplotlib.pyplot as plt
import scipy.io as sio

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

In [ ]:
# ── Carga de ECG sin ruido ─────────────────────────────────────
fs_ecg = 1000  # Hz
ecg = np.load('ecg_sin_ruido.npy').flatten()
t_ecg = np.arange(len(ecg)) / fs_ecg

# ── Carga de PPG sin ruido ─────────────────────────────────────
fs_ppg = 400  # Hz
ppg = np.load('ppg_sin_ruido.npy').flatten()
t_ppg = np.arange(len(ppg)) / fs_ppg

# ── Carga de Audio (La Cucaracha) ──────────────────────────────
fs_audio, audio_raw = sio.wavfile.read('la cucaracha.wav')

# Si el archivo es estéreo (multicanal), tomamos un solo canal o promediamos
# para no duplicar artificialmente la potencia total de la señal
if audio_raw.ndim > 1:
    audio = audio_raw[:, 0].astype(np.float64)
else:
    audio = audio_raw.astype(np.float64)

# Normalizamos la amplitud en [-1, 1]
audio = audio / np.max(np.abs(audio))
t_audio = np.arange(len(audio)) / fs_audio

print(f'ECG   -> {len(ecg)} muestras | Duración: {len(ecg)/fs_ecg:.1f} s | fs = {fs_ecg} Hz')
print(f'PPG   -> {len(ppg)} muestras | Duración: {len(ppg)/fs_ppg:.1f} s | fs = {fs_ppg} Hz')
print(f'Audio -> {len(audio)} muestras | Duración: {len(audio)/fs_audio:.1f} s | fs = {fs_audio} Hz')

Graficamos las señales en el tiempo para observar su comportamiento general antes de pasar al dominio frecuencial.

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(12, 8))
fig.suptitle('Señales en el dominio del tiempo', fontsize=13)

# ECG (mostramos los primeros 10 segundos)
seg_ecg = min(10 * fs_ecg, len(ecg))
axs[0].plot(t_ecg[:seg_ecg], ecg[:seg_ecg], color='steelblue', linewidth=0.8)
axs[0].set_title(f'ECG — Prueba de esfuerzo (fs = {fs_ecg} Hz)')
axs[0].set_xlabel('Tiempo [s]')
axs[0].set_ylabel('Amplitud [mV]')
axs[0].grid(True, alpha=0.4)

# PPG (mostramos el registro en reposo)
axs[1].plot(t_ppg, ppg, color='darkorange', linewidth=0.8)
axs[1].set_title(f'PPG — Registro en reposo (fs = {fs_ppg} Hz)')
axs[1].set_xlabel('Tiempo [s]')
axs[1].set_ylabel('Amplitud [u.a.]')
axs[1].grid(True, alpha=0.4)

# Audio (primeros 5 segundos)
seg_audio = min(5 * fs_audio, len(audio))
axs[2].plot(t_audio[:seg_audio], audio[:seg_audio], color='seagreen', linewidth=0.5)
axs[2].set_title(f'Audio — La Cucaracha (fs = {fs_audio} Hz)')
axs[2].set_xlabel('Tiempo [s]')
axs[2].set_ylabel('Amplitud norm.')
axs[2].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

## 2. Exploración de Parámetros de Welch

Para entender el impacto de la longitud de segmento `nperseg` en cada señal, comparamos tres casos:

- **Configuración ① (Segmento largo, $L = N/2$):** Proporciona la mejor resolución espectral posible pero con poquísimos segmentos para promediar ($K \approx 3$). El resultado tiene alta varianza y oscilaciones ruidosas.
- **Configuración ② (Óptima, $L$ moderado con overlap del 50%):** Logra un buen equilibrio: resolución suficiente para separar picos armónicos relevantes y una cantidad adecuada de promedios para que la curva sea suave y consistente.
- **Configuración ③ (Segmento muy corto):** Promedia una gran cantidad de segmentos haciendo que la curva sea muy suave, pero a costa de un lóbulo principal tan ancho que degrada la resolución y funde componentes cercanas.

In [ ]:
def welch_comparacion(señal, fs, configs, titulo, ax):
    for cfg in configs:
        f, Pxx = sig.welch(
            señal, fs=fs, window='hann',
            nperseg=cfg['nperseg'],
            noverlap=cfg['noverlap'],
            scaling='density'
        )
        step  = cfg['nperseg'] - cfg['noverlap']
        n_avg = 1 + (len(señal) - cfg['nperseg']) // step
        df    = fs / cfg['nperseg']
        Pxx_dB = 10 * np.log10(Pxx + 1e-12)
        label = f"{cfg['label']}  (K≈{n_avg}, Δf≈{df:.2f} Hz)"
        ax.plot(f, Pxx_dB, label=label, color=cfg['color'], linewidth=0.9)

    ax.set_title(titulo, fontsize=10)
    ax.set_xlabel('Frecuencia [Hz]')
    ax.set_ylabel('PSD [dB/Hz]')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.4)

In [ ]:
configs_ecg = [
    {'nperseg': len(ecg)//2, 'noverlap': len(ecg)//4, 'label': '① Pocos promedios (L=N/2)', 'color': 'crimson'},
    {'nperseg': 2048,        'noverlap': 1024,         'label': '② Óptima (L=2048)',          'color': 'royalblue'},
    {'nperseg': 128,         'noverlap': 64,           'label': '③ Excesiva (L=128)',          'color': 'forestgreen'},
]
configs_ppg = [
    {'nperseg': len(ppg)//2, 'noverlap': len(ppg)//4, 'label': '① Pocos promedios (L=N/2)', 'color': 'crimson'},
    {'nperseg': 1024,        'noverlap': 512,          'label': '② Óptima (L=1024)',          'color': 'royalblue'},
    {'nperseg': 64,          'noverlap': 32,           'label': '③ Excesiva (L=64)',           'color': 'forestgreen'},
]
configs_audio = [
    {'nperseg': len(audio)//2, 'noverlap': len(audio)//4, 'label': '① Pocos promedios (L=N/2)', 'color': 'crimson'},
    {'nperseg': 8192,          'noverlap': 4096,           'label': '② Óptima (L=8192)',          'color': 'royalblue'},
    {'nperseg': 256,           'noverlap': 128,            'label': '③ Excesiva (L=256)',          'color': 'forestgreen'},
]

fig_comp, axs_comp = plt.subplots(3, 1, figsize=(13, 12))
fig_comp.suptitle('Exploración del compromiso resolución vs. varianza en Welch', fontsize=13)

welch_comparacion(ecg,   fs_ecg,   configs_ecg,   f'ECG (fs={fs_ecg} Hz)',    axs_comp[0])
welch_comparacion(ppg,   fs_ppg,   configs_ppg,   f'PPG (fs={fs_ppg} Hz)',    axs_comp[1])
welch_comparacion(audio, fs_audio, configs_audio, f'Audio (fs={fs_audio} Hz)', axs_comp[2])

axs_comp[0].set_xlim(0, 100)
axs_comp[1].set_xlim(0, 20)
axs_comp[2].set_xlim(0, fs_audio // 2)

plt.tight_layout()
plt.show()

## 3. Estimación final de la PSD

A partir de la exploración realizada, se eligen los siguientes parámetros óptimos:
- **ECG:** `nperseg = 2048` $\implies \Delta f \approx 0.49\,\text{Hz}$ con unos 9 promedios. Resuelve con claridad la frecuencia cardíaca fundamental ($1-3\,\text{Hz}$) y sus armónicos sin ruido excesivo.
- **PPG:** `nperseg = 1024` $\implies \Delta f \approx 0.39\,\text{Hz}$ con $\approx 9$ promedios. Al ser una señal de pulso lento en reposo, este espaciado permite seguir los picos principales con buena definición.
- **Audio:** `nperseg = 8192` $\implies \Delta f \approx 5.38\,\text{Hz}$ (para $f_s=44.1\,\text{kHz}$). Brinda la resolución requerida para distinguir los componentes tonales de la melodía en el rango audible.

In [ ]:
f_ecg, P_ecg = sig.welch(
    ecg, fs=fs_ecg, window='hann',
    nperseg=2048, noverlap=1024, scaling='density'
)
f_ppg, P_ppg = sig.welch(
    ppg, fs=fs_ppg, window='hann',
    nperseg=1024, noverlap=512, scaling='density'
)
f_audio, P_audio = sig.welch(
    audio, fs=fs_audio, window='hann',
    nperseg=8192, noverlap=4096, scaling='density'
)

P_ecg_dB   = 10 * np.log10(P_ecg   + 1e-12)
P_ppg_dB   = 10 * np.log10(P_ppg   + 1e-12)
P_audio_dB = 10 * np.log10(P_audio + 1e-12)

## 4. Estimación del Ancho de Banda Efectivo

In [ ]:
def estimar_ancho_de_banda(f, P, porcentaje=0.995):
    """
    Calcula la frecuencia de corte hasta la cual se concentra el porcentaje
    especificado (por defecto 99.5%) de la energía total de la PSD.
    """
    potencia_acum = np.cumsum(P)
    potencia_total = potencia_acum[-1]
    idx_bw = np.searchsorted(potencia_acum / potencia_total, porcentaje)
    bw = f[min(idx_bw, len(f) - 1)]
    return bw, potencia_total

bw_ecg,   _ = estimar_ancho_de_banda(f_ecg,   P_ecg,   porcentaje=0.995)
bw_ppg,   _ = estimar_ancho_de_banda(f_ppg,   P_ppg,   porcentaje=0.995)
bw_audio, _ = estimar_ancho_de_banda(f_audio, P_audio, porcentaje=0.995)

print('ANCHO DE BANDA ESTIMADO (Criterio 99.5% de energía):')
print('─' * 70)
print(f'[ECG]        | BW = {bw_ecg:8.2f} Hz  (desde 0.00 Hz hasta {bw_ecg:.2f} Hz)')
print(f'[PPG]        | BW = {bw_ppg:8.2f} Hz  (desde 0.00 Hz hasta {bw_ppg:.2f} Hz)')
print(f'[Audio]      | BW = {bw_audio:8.2f} Hz  (desde 0.00 Hz hasta {bw_audio:.2f} Hz)')
print('─' * 70)

In [ ]:
fig_psd, axs_psd = plt.subplots(3, 1, figsize=(13, 12))
fig_psd.suptitle('PSD Final — Método de Welch con Ventana Hann', fontsize=13)

# ECG
axs_psd[0].plot(f_ecg, P_ecg_dB, color='steelblue', linewidth=1)
axs_psd[0].axvline(bw_ecg, color='red', linestyle='--', linewidth=1.2,
                   label=f'BW (99.5%) = {bw_ecg:.1f} Hz')
axs_psd[0].set_title(f'ECG  |  fs = {fs_ecg} Hz  |  L = 2048  |  Δf ≈ {fs_ecg/2048:.2f} Hz')
axs_psd[0].set_xlabel('Frecuencia [Hz]')
axs_psd[0].set_ylabel('PSD [dB/Hz]')
axs_psd[0].set_xlim(0, 100)
axs_psd[0].legend(fontsize=9)
axs_psd[0].grid(True, alpha=0.4)

# PPG
axs_psd[1].plot(f_ppg, P_ppg_dB, color='darkorange', linewidth=1)
axs_psd[1].axvline(bw_ppg, color='red', linestyle='--', linewidth=1.2,
                   label=f'BW (99.5%) = {bw_ppg:.1f} Hz')
axs_psd[1].set_title(f'PPG  |  fs = {fs_ppg} Hz  |  L = 1024  |  Δf ≈ {fs_ppg/1024:.2f} Hz')
axs_psd[1].set_xlabel('Frecuencia [Hz]')
axs_psd[1].set_ylabel('PSD [dB/Hz]')
axs_psd[1].set_xlim(0, 20)
axs_psd[1].legend(fontsize=9)
axs_psd[1].grid(True, alpha=0.4)

# Audio
axs_psd[2].plot(f_audio, P_audio_dB, color='seagreen', linewidth=0.8)
axs_psd[2].axvline(bw_audio, color='red', linestyle='--', linewidth=1.2,
                   label=f'BW (99.5%) = {bw_audio:.1f} Hz')
axs_psd[2].set_title(f'Audio (La Cucaracha)  |  fs = {fs_audio} Hz  |  L = 8192  |  Δf ≈ {fs_audio/8192:.1f} Hz')
axs_psd[2].set_xlabel('Frecuencia [Hz]')
axs_psd[2].set_ylabel('PSD [dB/Hz]')
axs_psd[2].set_xlim(0, fs_audio // 2)
axs_psd[2].legend(fontsize=9)
axs_psd[2].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

## 5. Tabla comparativa de resultados

In [ ]:
print('=' * 72)
print(f'{"TABLA RESUMEN — ANCHOS DE BANDA ESTIMADOS":<72}')
print('=' * 72)
print(f'{"Señal":<22} {"fs [Hz]":>9} {"L (nperseg)":>12} {"Δf [Hz]":>9} {"BW_99.5% [Hz]":>14}')
print('-' * 72)
print(f'{"ECG":<22} {fs_ecg:>9} {2048:>12} {fs_ecg/2048:>9.3f} {bw_ecg:>14.2f}')
print(f'{"PPG":<22} {fs_ppg:>9} {1024:>12} {fs_ppg/1024:>9.3f} {bw_ppg:>14.2f}')
print(f'{"Audio (La Cucaracha)":<22} {fs_audio:>9} {8192:>12} {fs_audio/8192:>9.2f} {bw_audio:>14.2f}')
print('=' * 72)
print('Criterio de BW: frecuencia acumulada al 99.5% de la energía total.')
print('Método: Welch con ventana Hann y 50% de solapamiento.')

## 6. Conclusiones

En el caso del **ECG de esfuerzo**, la mayor parte de la energía se encuentra concentrada por debajo de los $40\,\text{Hz}$, lo que es coherente con la fisiología cardíaca: el complejo QRS y las ondas P y T ocupan esa banda. El ancho de banda obtenido cercano a los $38\,\text{Hz}$ refleja fielmente esa distribución.

La señal **PPG en reposo** presenta un espectro notablemente más angosto y concentrado en frecuencias muy bajas. La frecuencia cardíaca basal se ubica alrededor de $1\,\text{Hz}$ (60 latidos por minuto) y la energía decae muy rápidamente después de los primeros armónicos, dando un ancho de banda efectivo muy inferior al del ECG.

Por último, el **Audio** contiene un contenido frecuencial distribuido en varios kilohercios debido a los instrumentos y armónicos de la melodía grabada, lo que justifica la necesidad de una frecuencia de muestreo del orden de decenas de kilohercios frente a las señales fisiológicas que solo requieren cientos de hercios.